# Predicting Coffee Yield from Agroecological and Landscape Features

By Adia Redd and Tiffany Tang

### Abstract

This study aimed to identify which agroecological or landscape feature best predicts mean coffee yield in Brazil, with the initial hypothesis that forest cover would be the most influential predictor. Using 51 environmental and landscape variables, we conducted exploratory analyses including histograms, correlation and principal component analysis to characterize data structure and multicollinearity. Preprocessing steps included label encoding, skewness-based log transformation, feature scaling and dimensionality reduction, which resulted in three dataset variants: a scaled/log-transformed, the same data set that was then PCA- transformed, and a reduced feature data set that was scaled, log transformed and PCA- transformed. 

We evaluated Linear Regression, Random Forest Regression and Support vector Regression using GridSearch CV and assessed performance with  R², error metrics, and diagnostic plots. Predictive performance was modest across all approaches, with the best models achieving test R² values of ~0.25–0.30. PCA and feature reduction generally reduced accuracy, suggesting that important signals were lost. SHAP analyses revealed that regional identifiers and temperature-related variables were the most influential predictors across models, while forest cover did not appear prominently in any model or principal component. Feature importance patterns varied across algorithms, limiting interpretability and model reliability.

Overall, results indicate that the current feature set and models are insufficient for accurately predicting coffee yield or identifying a dominant ecological driver. Future directions include exploring neural networks, gradient boosting, spatial modeling, and alternative dimensionality reduction methods to better capture complex environmental relationships.


### Introduction

Coffee production is a major global agricultural sector and Brazil is its largest producer, supplying roughly one-third of the world’s coffee. However, coffee yields can vary substantially across regions due to differences in climate, landscape structure, and ecological conditions. Prior ecological research has suggested that forest cover and landscape heterogeneity can enhance coffee productivity by improving pollination, stabilizing microclimates, and supporting natural pest control (De Marco & Coelho, 2004; Jha et al., 2014). These findings motivate a more quantitative assessment of how agroecological and landscape variables relate to yield across broader spatial scales.

The dataset used in this study (Silva et al., 2021; Zenodo record 5574892) combines municipal-level coffee yield with 51 environmental, climatic, and landscape features, including forest extent, temperature metrics, moisture indices, crop cover, topography, and spatial identifiers. Using this dataset, our goal is to answer the following research question: Which agroecological or landscape features best predict mean coffee yield across Brazilian farms? Based on previous ecological studies, our hypothesis is that forest cover will be the strongest predictor of yield.

Identifying the key drivers of coffee productivity is an important problem for both ecological and agricultural planning. Accurate prediction models can help farmers and policymakers anticipate yield constraints, evaluate the benefits of forested landscapes, and improve management under climate variability. To address this, we apply multiple machine learning models, Linear Regression, Random Forest Regression, and Support Vector Regression, along with extensive preprocessing, PCA, and SHAP-based interpretability to determine which variables consistently contribute to predictive performance.


### Methods

*Dataset and Research Design*

We used the publicly available agroecological dataset from Silva et al. (2021) (Zenodo record 5574892), which contains mean coffee yield and 51 landscape, climatic, and ecological variables from Brazilian municipalities. Our objective was to build predictive models of mean yield and determine which features contribute most strongly to prediction accuracy, with a specific hypothesis that forest cover would be a major predictor.

*Data Preprocessing*

Three categorical variables were label-encoded. Numerical features were screened for skewness using Pandas’ .skew(); variables with absolute skewness > 1.0 were considered for transformation, excluding binary, near-constant, and negative-valued variables. A natural log(+1) transformation was applied only when it reduced skewness. To avoid data leakage, all datasets were split into training and testing sets (80/20) before scaling or PCA.
Three preprocessing pipelines were evaluated:
Log-Transformed + Scaled Dataset
Log-Transformed + Scaled + PCA Dataset
Correlation/Variance-Reduced + Log-Transformed + PCA Dataset
 For Pipeline 3, features with correlation > 0.9 or variance < 0.01 were removed prior to PCA.

*Model Training*

We trained three families of regression models on each dataset variant: Ordinary Linear Regression; regularized models (Ridge, Lasso, ElasticNet); Random Forest Regression (RFR); and Support Vector Regression (SVR). Hyperparameters for all non-ordinary models were optimized using GridSearchCV with five-fold cross-validation. For RFR, the tree-splitting criterion and the maximum number of features considered at each split were tuned, with random_state=8 and out-of-bag error estimation enabled. For the SVR model, we performed a grid search over the C, gamma, and epsilon hyperparameters to identify the best-performing configuration of the RBF kernel. The RBF kernel was selected because it can model non linear relationships, which are expected in ecological yield data.

*Model Evaluation*

Performance was assessed using R², mean absolute error (MAE), and root mean squared error (RMSE), which are standard metrics for regression prediction. Residual plots and QQ plots were generated to evaluate model assumptions and residual behavior. Feature importance and interpretability were assessed using SHAP values for each best-performing model on each dataset variant.

*Reproducibility*

All analyses were conducted in Python. Full preprocessing, modeling and visualization code is included below in Appendix A to ensure complete reproducibility. 

### Results

#### *Exploratory Data Analysis*

Initial visualization of all 51 agroecological and landscape features (Fig. 1)revealed substantial non-normality across many predictors. Several variables exhibited heavy right skew, prompting the use of log-transformations to reduce extreme values and improve model stability. A log(x + 1) transformation was applied to the subset of highly skewed continuous features (Fig. 1). Although this transformation shifted some variables toward more symmetric distributions, most remained moderately or strongly skewed (Fig. 3a–b). Overall, the log transformation offered only partial improvement and did not meaningfully change the overall distributional structure of the dataset.


<div style="text-align:center; margin-bottom:30px;">
  <img src="Figures_Report/Figure1.png" width="600">
  <p><strong>Fig 1.</strong> Feature Distribution Histograms</p>
</div>

<div style="text-align:center; margin-bottom:30px;">
  <img src="Figures_Report/Figure6.png" width="800">
  <p><strong>Fig 2.</strong> Output of Skewed Columns</p>
</div>

<div style="display:flex; justify-content:center; gap:20px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure7a.png" width="350">
    <p><strong>Fig 3a.</strong> Some Features Before Log Transformation</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure7b.png" width="350">
    <p><strong>Fig 3b.</strong> Some Features After Log Transformation</p>
  </div>

</div>



A correlation heatmap (Fig. 2) revealed three major multicollinearity clusters:
(1) vegetation and field-cover metrics, (2) crop-cover indicators, and (3) temperature-related variables.
These clusters indicate substantial redundancy among predictors that likely capture similar underlying ecological processes. This motivated the creation of a PCA-reduced dataset as well as a reduced-feature dataset where highly correlated variables and features with 0 variance were removed upfront before being PCA transformed.

<div style="text-align:center; margin-bottom:30px;">
  <img src="Figures_Report/Figure2.png" width="350">
  <p><strong>Fig 4.</strong> Correlation Heatmap</p>
</div>


Principal Component Analysis (PCA) was performed on the scaled dataset to evaluate underlying structure. In Figure 5 we see that The first 20 principal components explained approximately 95% of total variance, with an elbow at three components (55% cumulative variance). However, initial modeling showed that three components were insufficient; therefore, the first 7 PCs (75% cumulative variance) were retained for PCA-based models. Scatterplots of samples projected onto PC1–PC2 (Fig. 6) and PC1–PC3 (Fig. 7)showed no visible clustering by average yield, suggesting limited separability of yield levels in a linear subspace.

<div style="display: flex; justify-content: space-around;">

  <div style="text-align: center;">
    <img src="Figures_Report/Figure3.png" width="300">
    <p><strong>Fig 5.</strong> Cumulative Variance</p>
  </div>

  <div style="text-align: center;">
    <img src="Figures_Report/Figure4.png" width="300">
    <p><strong>Fig 6.</strong> PC1 vs PC2 Scatter</p>
  </div>

  <div style="text-align: center;">
    <img src="Figures_Report/Figure5.png" width="300">
    <p><strong>Fig 7.</strong> 3D PCA Scatter</p>
  </div>

</div>

To evaluate the effect of preprocessing and dimensionality reduction on predictive performance, three dataset variants were constructed:

First Dataset (No PCA)

- This version retained all original 51 predictors after scaling and selective log transformation. It served as the baseline for comparison and allowed for direct interpretation of feature coefficients and importance scores.

Second Dataset (7 PCs)

- Principal Component Analysis was applied to the scaled data (after selective log transformation), and the first seven principal components—explaining approximately 75% of the total variance—were selected (Figs. 3–4). This representation aimed to reduce collinearity and compress the environmental signal into a smaller set of orthogonal predictors.

Third Dataset

- Highly correlated (>0.9) and near-zero-variance predictors were removed prior to scaling and PCA. This version evaluates whether feature pruning followed by dimensionality reduction yields more stable or interpretable models.

All models were trained on an 80/20 train–test split using standardized inputs for all non-tree algorithms. This ensured a fair, consistent comparison across approaches while preserving proper separation of training, validation, and test data.

#### *Linear Regression*

*Non-PCA Model*

Ordinary Linear Regression achieved a test R² of approximately 0.11, indicating that the raw feature space provides very limited linear signal for predicting coffee yield. Introducing regularization improved cross-validated R² values: Ridge, Lasso, and ElasticNet models reached ~0.20–0.21 during training, but their performance declined on the held-out test set (test R² ≈ 0.12–0.15). This pattern suggests mild overfitting, where regularization helps stabilize coefficients but cannot uncover strong linear structure in the data (Fig. 8).

<div style="text-align:center; margin-bottom:30px;">
  <img src="Figures_Report/Figure8.png" width="350">
  <p><strong>Fig 8.</strong> Linear Regression First Dataset Output</p>
</div>


*PCA Models*

Applying PCA did not improve linear model performance. When trained on the first seven principal components, which together captured approximately 75% of the variance, all linear models, including regularized variants, performed slightly worse than their non-PCA counterparts. Test R² values generally fell below 0.10, indicating that dimensionality reduction removed variance that was weakly but genuinely related to yield, without revealing any clearer linear structure.

These results suggest that although PCA effectively compresses the feature space, the principal components do not align with directions that support linear prediction of coffee yield. In other words, the dominant sources of variance in the landscape and ecological data are not the same sources of variance that drive yield differences.


<div style="text-align:center; margin-bottom:30px;">
  <img src="Figures_Report/Figure9.png" width="350">
  <p><strong>Fig 9.</strong> Linear Regression Second Dataset Output</p>
</div>


When highly correlated and near-zero variance features were removed prior to PCA (third dataset), linear model performance deteriorated substantially. Across all regularized and unregularized variants, test R² values were near zero or negative, indicating that the models performed worse than a baseline predictor that simply outputs the mean yield. This pattern suggests that the feature-reduction step discarded variables containing meaningful but diffuse signal, leaving the resulting principal components unable to represent relationships relevant for linear prediction. As a result, the models exhibited severe underfitting, reinforcing that linear methods are highly sensitive to information loss introduced during aggressive preprocessing.

<div style="text-align:center; margin-bottom:30px;">
  <img src="Figures_Report/Figure10.png" width="350">
  <p><strong>Fig 10.</strong> Linear Regression Third Dataset Output</p>
</div>




*SHAP Feature Importance*

SHAP analysis of the linear regression model trained on the full, non-PCA dataset revealed a small number of moderately influential variables (Fig. 11). codigo_ibg, a geographic identifier, showed the strongest effect on predicted yield, suggesting substantial spatial structure not fully captured by the agroecological features. Temperature-related variables—including isoterm.mean, temp_season.mean, and temp_range.mean—also contributed meaningfully but exhibited mixed positive and negative SHAP values, indicating that their relationships with yield are likely nonlinear or interactive. Most other features clustered tightly around zero, consistent with the model’s overall low predictive power (R² ≈ 0.11–0.20). Forest cover variables were notably absent from the top contributors, contrary to the original hypothesis.

SHAP results from the PCA-transformed linear model demonstrate a shift in feature importance from raw environmental variables to principal components (Fig. 12). PC2 emerged as the strongest predictor, displaying the largest spread in SHAP values and the clearest separation between high- and low-valued observations. PC3, PC6, and PC4 also contributed moderately, whereas PC1—which explains the greatest variance in the raw feature space—had minimal impact on yield prediction.

This pattern indicates that the principal components capturing the most overall variance are not necessarily those containing the variance relevant for predicting yield. Specifically, PC2 carries a mix of temperature, precipitation, and pollination-related variation, placing it closer to the underlying ecological processes associated with productivity. However, despite PC2’s prominence, SHAP values remained small in magnitude overall, consistent with the model’s reduced predictive performance after PCA (test R² ≈ 0.15–0.20).

After removing highly correlated and near-zero variance features prior to PCA, the SHAP importance patterns changed substantially (Fig 13). PC1 became more influential than in the full-PCA model, reflecting the reshaped structure of the feature space after dimensionality reduction. PC2 and PC3 remained moderately important, but overall SHAP magnitudes were noticeably smaller than in the other datasets, consistent with the model’s severe underfitting and negative R² values on the test set.

These results suggest that the process of removing correlated features eliminated meaningful predictive signal, causing the resulting principal components to become less aligned with yield-driving ecological variation. The reduced PCA space therefore captured less biologically relevant structure, and the linear regression model was unable to recover predictive relationships.


<div style="display:flex; justify-content:center; gap:40px; margin-bottom:30px;">

  
  <div style="text-align:center;">
    <img src="Figures_Report/Figure11.png" width="400">
    <p><strong>Fig 11.</strong> SHAP Summary Plot First Dataset</p>
  </div>

  
  <div>
    <div style="text-align:center; margin-bottom:25px;">
      <img src="Figures_Report/Figure12.png" width="350">
      <p><strong>Fig 12.</strong> SHAP Summary Plot Second Dataset</p>
    </div>
    <div style="text-align:center;">
      <img src="Figures_Report/Figure13.png" width="350">
      <p><strong>Fig 13.</strong> SHAP Summary Plot Third Dataset</p>
    </div>

  </div>

</div>


*Residuals and QQ plot Diagnostics*

Residual diagnostics were used to evaluate linear regression assumptions across the three preprocessing approaches: (1) log-transformed and scaled predictors, (2) PCA-transformed predictors, and (3) reduced-feature PCA predictors. The residual–predicted plots (Fig. 14a–c) and QQ plots (Fig. 15a–c) consistently reveal substantial deviations from the assumptions required for reliable linear modeling.

The three residual plots show a largely patternless cloud, indicating no strong violations of linear regression assumptions (e.g., heteroscedasticity or nonlinearity). However, this randomness also reflects the fact that the models are underfitting: the predicted values vary very little while the residuals remain large. In other words, the model is not capturing meaningful structure in the data, which produces ‘clean’ residual patterns but poor predictive performance.The residuals remain widely dispersed around zero, and no model shows evidence of improved homoscedasticity following transformation or PCA. In the PCA-based models (Figs. 14b and 14c), the predicted-value range collapses compared to the non-PCA model, reflecting the loss of variance introduced by dimensionality reduction; however, this compression does not generate more well-behaved residuals. Instead, the spread remains large relative to the narrow band of predictions, indicating underfitting.

The QQ plots (Figs. 15a–c) show systematic departures from normality in all cases. While the middle quantiles roughly follow the theoretical line, the tails diverge substantially—particularly the upper tail—which indicates non-normal, heavy-tailed residuals. Neither log-scaling nor PCA succeeded in producing residual distributions resembling a Gaussian, further confirming model misspecification. The reduced-feature PCA model exhibits the largest tail deviations, consistent with its poorest predictive performance.

The residual and QQ plot diagnostics reinforce the quantitative model metrics: linear regression assumptions are violated under all preprocessing approaches, with clear evidence of nonlinearity, heteroscedasticity, and heavy-tailed error structure. These findings confirm that linear regression is poorly suited for modeling mean coffee yield in this dataset and that more flexible nonlinear approaches (e.g., SVR, Random Forest) are better aligned with the underlying data structure.

<div style="display:flex; justify-content:center; gap:25px; margin-bottom:30px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure14a.png" width="300">
    <p><strong>Fig 14a.</strong> Linear Regression First Dataset Residuals Plot</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure14b.png" width="300">
    <p><strong>Fig 14b.</strong> Linear Regression Second Dataset Residuals Plot</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure14c.png" width="300">
    <p><strong>Fig 14c.</strong> Linear Regression Third Dataset Residuals Plot</p>
  </div>

</div>

<div style="display:flex; justify-content:center; gap:25px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure15a.png" width="300">
    <p><strong>Fig 15a.</strong> Linear Regression First Dataset QQ Plot</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure15b.png" width="300">
    <p><strong>Fig 15b.</strong> Linear Regression Second Dataset QQ Plot</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure15c.png" width="300">
    <p><strong>Fig 15c.</strong> Linear Regression Third Dataset QQ Plot</p>
  </div>

</div>



#### *Random Forest Regression*

*Non PCA Model*

The best model of all three families is the RFR trained on the data that was simply log-transformed and scaled (dataset variant 1), defined the loss function as squared error and the maximum number of features to be considered when making the split in a given tree equal to the log base 2 of the number of samples (Fig 16a). The R² score of this model with the test set is 0.28 (Fig 16b).


<div style="display: flex; justify-content: center; gap:20px;">
    <div style="margin: 5px; width="300">
        <img src="Figures_Report/Figure16a.png" alt="RFR non-PCA model stats GS" style="width: 100%; height: auto;">
        <p><strong>Fig 16a.</strong> RFR Dataset Variant 1 GridSearchCV Output</p>
    </div>
    <div style="margin: 5px; gap:15px">
        <img src="Figures_Report/Figure16b.png" alt="RFR non-PCA model stats test" style="width: 100%; height: auto;">
        <p><strong>Fig 16b.</strong> RFR Dataset Variant 1 Model Test Output</p>
    </div>
</div>

*PCA Models*

GridSearch reported the same optimal model when using the remaining two dataset variants for fitting, where the loss function is absolute error and the maximum number of features to be considered when making the split equal to the square root of the number of samples (Fig 17a, 18a). This model returned R² scores of 0.28 and -0.05 (Fig 17b, 18b) when tasked with predicting the mean yield using their respective test sets.

<div style="display: flex; justify-content: center; gap:20px;">
    <div style="margin: 5px;">
        <img src="Figures_Report/Figure17a.png" alt="RFR non-PCA model stats GS" style="width: 100%; height: auto;">
        <p><strong>Fig 17a.</strong> RFR Dataset Variant 2 GridSearchCV Output</p>
    </div>
    <div style="margin: 5px;">
        <img src="Figures_Report/Figure18a.png" alt="RFR non-PCA model stats test" style="width: 100%; height: auto;">
        <p><strong>Fig 18a.</strong> RFR Dataset Variant 2 GridSearch CV Output</p>
    </div>
</div>

<div style="display: flex; justify-content: center; gap:20px;">
    <div style="margin: 5px;">
        <img src="Figures_Report/Figure17b.png" alt="RFR non-PCA model stats GS" style="width: 100%; height: auto;">
        <p><strong>Fig 17b.</strong> RFR Dataset Variant 2 Model Test Output</p>
    </div>
    <div style="margin: 5px;">
        <img src="Figures_Report/Figure18b.png" alt="RFR non-PCA model stats test" style="width: 100%; height: auto;">
        <p><strong>Fig 18b.</strong> RFR Dataset Variant 2 Model Test Output</p>
    </div>
</div>


*Residuals and QQ plots*

Despite this poor correlation between the predicted and actual y-values, the residuals plots did not indicate any extremely abnormal behavior for either model. The residuals produced by the top scoring model did, however, display a very mild positive correlation between the actual and predicted mean yield (Fig 19a). This could indicate that the dataset could benefit from a more complex model to explain the relationship between the features and the mean yield since there may be a violation of the assumptions associated with regression. The remaining two models produced residual plots with no discernible pattern (Fig 19b, 19c).

<div style="display:flex; justify-content:center; gap:25px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure19a.png" width="300">
    <p><strong>Fig 19a.</strong> Residual Plot of RFR Model Predictions on Dataset Variant 1</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure19b.png" width="300">
    <p><strong>Fig 19b.</strong> Residual Plot of RFR Model Predictions on Dataset Variant 2</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure19c.png" width="300">
    <p><strong>Fig 19c.</strong> Residual Plot of RFR Model Predictions on Dataset Variant 3</p>
  </div>

</div>

To confirm the normality of the errors, the residuals of each model were plotted on a QQ-plot, where it became apparent thar none the residuals were not normally distributed, despite what the actual residual plots themselves may have suggested (Fig 20a, 20b, 20c).

<div style="display:flex; justify-content:center; gap:25px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure20a.png" width="300">
    <p><strong>Fig 20a.</strong> QQ Plot of RFR Model Predictions on Dataset Variant 1</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure20b.png" width="300">
    <p><strong>Fig 20b.</strong> QQ Plot of RFR Model Predictions on Dataset Variant 2</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure20c.png" width="300">
    <p><strong>Fig 20c.</strong> QQ Plot of RFR Model Predictions on Dataset Variant 3</p>
  </div>

</div>

*SHAP Feature Importance*

SHAP summaries were generated for both models to investigate which feature was the most important in the model's decision to predict a value for a given sample. The first model's SHAP summary (Fig 21) reported the top five most important features to all be temperature-related, aside from argotox_P, which is the percentage of farms in the municipality that used the ArgoTox plant protection agent.

<div style="display:flex; justify-content:center; gap:40px; margin-bottom:30px;">
  <div style="text-align:center;">
    <img src="Figures_Report/Figure21.png" width="400">
    <p><strong>Fig 21.</strong> RFR SHAP Summary for Dataset Variant 1</p>
  </div>

</div>


For all features there is no clear separation of high and low values in terms of SHAP value. This indicates that for most samples, the model had a difficult time partitioning a sample into one node or the other when a split in a tree arose because it could not be determined if a high or low value was important or not. When predicting y-values for the second dataset variant's test split, the second model selected PC2 as it's most important feature and PC1 as it's least important (Fig 22). When predicting the y-values for the third dataset variant's test split, PC1 was selected as most important and PC7 as least important. Despite being the same model, the important components changes drastically (Fig 23). Since PC2 was indicated a number of times across the models, it will be discussed separately. PC1 is composed mostly of geographic features like Euclidean distance between forest fragments and coffee fields, longitude/latitude, and municipality identifiers.

<div style="display:flex; justify-content:center; gap:25px; margin-bottom:30px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure12.png" width="350">
      <p><strong>Fig 22.</strong> RFR SHAP Summary for Dataset Variant 2</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure23.png" width="350">
      <p><strong>Fig 23.</strong> RFR SHAP Summary for Dataset Variant 3</p>
  </div>

</div>

#### *Support Vector Regression*

*Non-PCA Models*

Among the SVR configurations evaluated, the non-PCA model trained on the first dataset achieved the strongest overall performance, with an R² of approximately 0.25 on the training set and 0.28 on the test set (Fig. 24a–b). The similarity between training and test performance indicates stable generalization and little evidence of overfitting. Corresponding error metrics (MAE ≈ 4.39; RMSE ≈ 5.73) suggest moderate predictive accuracy, indicating that SVR was able to capture some nonlinear relationships between agroecological variables and mean coffee yield.

<div style="display:flex; justify-content:center; gap:25px; margin-bottom:30px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure24a.png" width="350">
      <p><strong>Fig 24a.</strong> SVR First Dataset Training Output</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure24b.png" width="350">
      <p><strong>Fig 24b.</strong> SVR First Dataset Testing</p>
  </div>

</div>

*PCA Models*

Applying SVR to PCA-transformed inputs resulted in a modest decline in predictive performance, with R² values ranging from approximately 0.16 to 0.24 (Fig. 25a–b). Although PCA reduced dimensionality and multicollinearity, restricting the model to seven principal components likely removed predictive structure relevant to yield. Additionally, the nonlinear relationships that SVR is designed to model may not be fully preserved in the orthogonal PCA space, limiting its effectiveness.

<div style="display:flex; justify-content:center; gap:25px; margin-bottom:30px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure25a.png" width="350">
      <p><strong>Fig 25a.</strong> SVR Second Dataset Training Output</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure25b.png" width="350">
      <p><strong>Fig 25b.</strong> SVR Second Dataset Testing output</p>
  </div>

</div>

SVR performance declined substantially when correlated features were removed prior to PCA, resulting in negative test-set R² values (Fig. 26a–b). This indicates that the model performed worse than a baseline mean predictor on unseen data. As observed in other modeling approaches, aggressive feature reduction appears to have eliminated informative signal, suggesting that correlated variables in this dataset still contain meaningful predictive information rather than purely redundant noise.

<div style="display:flex; justify-content:center; gap:25px; margin-bottom:30px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure26a.png" width="350">
      <p><strong>Fig 26a.</strong> SVR Third Dataset Training Output</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure26b.png" width="350">
      <p><strong>Fig 26b.</strong> SVR Third Dataset Testing output</p>
  </div>

</div>



*Residual Plots*

Residual plots were examined to assess model fit, bias, and systematic prediction errors across the three SVR configurations: non-PCA inputs, PCA-transformed inputs, and reduced-feature PCA inputs (Fig. 27a–c).

For the non-PCA SVR model, residuals were broadly centered around zero across most predicted yield values, indicating an overall unbiased fit (Fig. 27a). However, a slight downward trend in the smoothed residual curve at higher predicted yields suggests mild underprediction for municipalities with higher mean coffee yield. The dispersion of residuals increases modestly at mid-range predictions, indicating some heteroscedasticity, though no extreme pattern of systematic error is evident.

In the PCA-based SVR model, residuals exhibited greater spread and a more pronounced nonlinear trend (Fig. 27b). The smoothed curve shows alternating regions of over- and underprediction, particularly in the mid-to-high predicted yield range, suggesting that the PCA transformation distorted some of the nonlinear relationships between predictors and yield. While residuals remain roughly centered near zero overall, the increased variability reflects reduced predictive stability compared to the non-PCA model.

The reduced-feature PCA SVR model showed the weakest residual structure (Fig. 27c). Residuals display increased scatter and a clear downward trend at higher predicted yields, indicating consistent underprediction for high-yield observations. This pattern aligns with the negative R² values observed for this configuration and suggests that aggressive feature reduction removed informative signal necessary for accurate prediction. The absence of a flat residual trend further indicates poor model calibration across the prediction range.

<div style="display:flex; justify-content:center; gap:25px; margin-bottom:30px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure27a.png" width="300">
    <p><strong>Fig 27a.</strong> SVR First Dataset Residuals Plot</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure27b.png" width="300">
    <p><strong>Fig 27b.</strong> SVR Second Dataset Residuals Plot</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure27c.png" width="300">
    <p><strong>Fig 27c.</strong> SVR Third Dataset Residuals Plot</p>
  </div>


*SHAP Feature Importance*

SHAP (SHapley Additive exPlanations) values were used to interpret feature contributions to SVR predictions across the non-PCA, PCA-transformed, and reduced-feature PCA models (Fig. 30-32). 

For the SVR model trained on the full feature set, SHAP summaries indicate that geographic identifiers and climatic variables were the dominant contributors to model predictions (Fig. 30). In particular, codigo_ibg (municipality identifier), isoterm.mean, mean temperature, and precipitation-related variables exhibited the largest SHAP magnitudes, indicating strong influence on predicted yield. Higher values of temperature-related features were generally associated with positive SHAP values, suggesting increased predicted yield under warmer conditions.

Landscape and land-use variables, including coffee cover (CC_ha), patch density, and coffee agrotoxic usage, showed smaller but non-negligible contributions. The relatively broad spread of SHAP values across features suggests that the model relied on a combination of climatic, geographic, and land-use information rather than a single dominant predictor.

When SVR was applied to PCA-transformed inputs, SHAP values reflected the dominance of the leading principal components, particularly PC1 and PC2 (Fig. 31). These components exhibited the largest absolute SHAP values, indicating that most predictive signal was captured by the first two axes of variation in the data. Lower-order components (PC3–PC7) contributed minimally to model output.

In the reduced-feature PCA model, SHAP values were more uniformly compressed toward zero, with PC1 and PC2 still dominating but with reduced magnitude (Fig. 32). This attenuation of SHAP values aligns with the poor predictive performance observed for this configuration and indicates that aggressive feature filtering prior to PCA substantially weakened the available predictive signal.

The diminished contribution of higher-order components and the overall narrowing of SHAP distributions suggest that key information necessary for differentiating high- and low-yield municipalities was lost during feature reduction. As a result, the model relied on a limited and weakened representation of the original data structure.


Across all SVR configurations, SHAP analysis consistently highlights the importance of spatial and climatic variation in driving coffee yield predictions. Models trained on the full feature set retained richer and more interpretable feature contributions, while PCA-based models increasingly obscured or eliminated meaningful structure. These findings reinforce earlier performance results, demonstrating that dimensionality reduction, particularly when combined with feature elimination, can hinder both predictive accuracy and interpretability in this dataset.


<div style="display:flex; justify-content:center; gap:40px; margin-bottom:30px;">

  
  <div style="text-align:center;">
    <img src="Figures_Report/Figure30.png" width="400">
    <p><strong>Fig 28.</strong> SVR SHAP Summary Plot First Dataset</p>
  </div>

  
  <div>
    <div style="text-align:center; margin-bottom:25px;">
      <img src="Figures_Report/Figure31.png" width="350">
      <p><strong>Fig 29.</strong> SVR SHAP Summary Plot Second Dataset</p>
    </div>
    <div style="text-align:center;">
      <img src="Figures_Report/Figure32.png" width="350">
      <p><strong>Fig 30.</strong> SVR SHAP Summary Plot Third Dataset</p>
    </div>

  </div>

</div>


#### *Principal Component 2*

Since PC2 was repeatedly revealed as an important feature we investigated which features were used to create it. In the full feature PCA, PC2 is dominated by temperature-related variables, along with average pollination activity and latitude, which all load positively (Fig 33a). Elevation, warm-season average temp, and temperature range load negatively. After removing correlated features, the structure of PC2 shifts: elevation becomes the strongest driver, loading positively, and is accompanied by warm season precipitation-related variables, while several cool-season precipitation-related variables load negatively (Fig 33b).
Both PC2’s contain temperature features, but neither load forest cover into their composition.

<div style="display:flex; justify-content:center; gap:25px;">

  <div style="text-align:center;">
    <img src="Figures_Report/Figure33a.png" width="300">
    <p><strong>Fig 33a.</strong> PC2 loading of Dataset Variant 2</p>
  </div>

  <div style="text-align:center;">
    <img src="Figures_Report/Figure33b.png" width="300">
    <p><strong>Fig 33b.</strong> PC 2 loadinf of Dataset Variant 3</p>
  </div>

### Conclusion and Discussion

The objective of this study was to evaluate whether agroecological and landscape features could reliably predict mean coffee yield across Brazilian municipalities and to test the hypothesis that forest cover would be the most influential predictor. Overall, the results indicate that the models evaluated were unable to achieve strong or consistent predictive performance, limiting their utility for identifying a single dominant driver of yield variation.

Across all modeling approaches, predictive accuracy was modest, with the best-performing models achieving test-set R² values of approximately 0.25–0.30. While nonlinear models such as Support Vector Regression and Random Forest Regression outperformed linear models, their performance remained limited. Residual diagnostics suggested that errors from the nonlinear models were approximately symmetric; however, formal normality assessment using QQ-plots was only conducted for the linear regression models. For these models, departures from normality indicate that linear model assumptions were not fully satisfied and that important structure in the data may remain unexplained. Linear regression models produced the lowest R² values, indicating that linear relationships alone are insufficient to capture the complexity of yield dynamics.

Interpretability analyses using SHAP further support these findings. Although several features, particularly geographic identifiers, temperature-related variables, and selected principal components, were identified as influential in certain models, these effects were not consistent across modeling approaches or dataset variants. Importantly, forest cover did not emerge as a dominant predictor in any of the models evaluated. In the original feature space, forest cover showed minimal contribution to model output, and in PCA-based models, it did not substantially load onto the principal components that were most influential for prediction. As a result, there is insufficient evidence to support the hypothesis that forest cover is the primary driver of mean coffee yield in this dataset.

Several limitations may explain these results. First, coffee yield is likely influenced by complex, nonlinear interactions among climatic, management, and socioeconomic factors that are difficult to capture using municipality-level aggregate data. Second, strong spatial structure in the data, evidenced by the high SHAP importance of geographic identifiers such as codigo_ibg, may mask underlying causal relationships, leading models to rely on location-based proxies rather than mechanistic drivers. Finally, dimensionality reduction techniques such as PCA, while useful for mitigating multicollinearity, may remove meaningful ecological signal, particularly when applied aggressively or in conjunction with feature filtering.

Future work could address these limitations by incorporating higher-resolution spatial or temporal data, explicitly modeling spatial dependence, or exploring interaction terms and hierarchical modeling approaches. Additionally, future analyses could begin by evaluating deep learning techniques, which may be better suited to capturing complex nonlinear relationships and high-dimensional interactions present in agroecological datasets. Incorporating farm-level management variables or time-series yield data may further improve predictive performance and allow for a more nuanced assessment of how forest cover interacts with climatic and landscape factors. Such extensions would be necessary to more conclusively evaluate the role of forest cover in shaping coffee yield outcomes.


### References

González-Chaves, Adrian, et al. “Positive Forest Cover Effects on Coffee Yields Are Consistent Across Regions.” Zenodo (CERN European Organization for Nuclear Research), 8 Oct. 2021, https://doi.org/10.5061/dryad.612jm644g.

Yadav, Amit. “SHAP Values vs Feature Importance.” Medium, 19 Sept. 2024, medium.com/biased-algorithms/shap-values-vs-feature-importance-ba6b91c16319. 

Kusawa, Sunny, and Sunny Kusawa. “When to Perform Feature Scaling? Before Dataset Split or After ? And Why?” Data Magic AI Blog, 26 July 2023, datamagiclab.com/when-to-perform-feature-scaling. 


## Appendix A

In [ ]:
# import libraries
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import tqdm as notebook_tqdm
import shap
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import statsmodels.api as sm

In [ ]:
# read in the data and take a look at it
# change path to fit your needs
file_path = r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\cleaned_coffee.csv"

#file_path = r'/Users/adia/PyCharmMiscProject/ML/Final/cleaned_coffee.csv'

# Read the CSV
df = pd.read_csv(file_path, sep = ';') # semicolon delimited file

# Take a quick look at the data
print(df.head())

### Exploratory Analysis

In [ ]:
# Label Encoding
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
categorical_cols
df_encoded = df.copy() # copy the dataframe

label = LabelEncoder() # label encode each column that is non numeric
for col in categorical_cols:
    df_encoded[col] = label.fit_transform(df_encoded[col].astype(str))

df_encoded = df_encoded.drop(columns=["ID"]) # don't need the ID column
# save this
df_encoded.to_csv("df_encoded.csv", index=False)

In [ ]:
# Exploratory analysis
# Correlation
corr = df_encoded.corr()
corr.head() # just to take a look
plt.figure(figsize=(14, 12))# set figure size
sns.heatmap(corr, cmap= "coolwarm", annot= False)
plt.title("Correlation Heatmap of the Label Encoded Data")
#plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\correlation_heatmap.png")
plt.show()

# Feature histogram
df_encoded.hist(figsize= (30,30))
plt.title("Feature Distributions")
#plt.savefig(r'/Users/adia/PyCharmMiscProject/ML/Final/Figures/feature_dist_histograms.png')
plt.show()

In [ ]:
# Log transform skewed features
# function to detect skewness- log transformations cannot change left skew!
def detect_skewed_features(df, skew_threshold=1.0):
    skewed_features = []

    for col in df.columns:
        series = df[col]
        # Skip binary or near-constant
        if series.nunique() <= 3:
            continue
        # Skip negative values (log1p cannot handle)
        if (series < 0).any():
            continue
        # Original skewness
        orig_skew = series.skew()
        # Must be skewed enough to consider transforming
        if abs(orig_skew) <= skew_threshold:
            continue
        # Skewness after log1p
        log_skew = np.log1p(series).skew()
        # Only transform if log reduces skewness
        if abs(log_skew) < abs(orig_skew):
            skewed_features.append(col)

    return skewed_features
# let's see the skewed columns
skewed_cols = detect_skewed_features(df_encoded)

# just in case keep the non-skewed columns separately
non_skewed_cols = [c for c in df_encoded.columns if c not in skewed_cols]

print("Skewed columns to log-transform:")
print(skewed_cols)
len(skewed_cols)

# apply log1p to skewed features
df_log = df_encoded.copy() # copy the dataset over

df_log[skewed_cols] = np.log1p(df_log[skewed_cols])

# define X and y, we will be using these from here on forward!!!!
y = df_log["yield.mean"]
X = df_log.drop(columns=["yield.mean"])

# lets replot this histogram to see if the feature skew is better
df_log.iloc[:,:52].hist(figsize= (30,30))
plt.title("Feature Distributions After Log Transformation")
plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\feature_log_hist.png")
plt.show()

In [ ]:
# Exploratory Analysis
# PCA, using the encoded and log transformed dataset 
# first scale, only want to look at the predictors
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# save as a dataframe
pca_columns = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pca_columns)

# Add the target column back
# This is for our visualization plots, it was not used to compute the PC's
df_pca["yield.mean"] = y.values

df_pca.plot(kind='scatter', x='PC1', y='PC2', c= df_pca["yield.mean"],  cmap='viridis')
plt.title('Scatter Plot of Data Using PC1/PC2')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True)
#plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\pca_scatter.png")
plt.show()

# 3D plot
from mpl_toolkits.mplot3d import Axes3D

# assign the definitions of each quartile
lowbound = df_pca['yield.mean'].min()
q1 = df_pca['yield.mean'].quantile(.25)
q2 = df_pca['yield.mean'].quantile(.5)
q3 = df_pca['yield.mean'].quantile(.75)
highbound = df_pca['yield.mean'].max()

# separate by quartile function
def assign_quartile(value):
    if value <= q1:
        return 0
    elif value <= q2:
        return 1
    elif value <= q3:
        return 2
    else:
        return 3

colors = df_pca['yield.mean'].apply(assign_quartile) # color code the quartiles

# plot

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
scatplt = ax.scatter(df_pca['PC1'], df_pca['PC2'], df_pca['PC3'], c=colors, cmap='viridis',s=5)

cbar = plt.colorbar(scatplt, ax=ax, ticks=[0, 1, 2, 3],shrink=0.4, pad=.1)
cbar.set_label('Yield Quartile')
cbar.set_ticklabels(['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])

ax.set_xlabel('PC1')
ax.set_ylabel('PC2',labelpad=10)
ax.set_zlabel('PC3')
ax.set_title('Samples colored by Yield Mean Quartiles on the top 3 PC Axes')

#plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\pca_scatter3D.png")
plt.show()

# see if any of the PC's have a correlation with the yield mean
# Compute correlation between each PC and the target
pc_target_corr = df_pca.drop("yield.mean", axis=1).corrwith(df_pca["yield.mean"])
pc_target_corr
# output was not promising

# total variance plots
plt.figure(figsize=(10,5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o') # plot the variance ratio
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained by PCA")
plt.grid(True)
#plt.savefig(r"C:\Users\tiffa\Machine_Learning_Final\ML\Final\Figures\pca_cumulative_var.png")
plt.show()

### Data Processing

In [ ]:
# Prepare out datasets to be used
# Split and scale
# train, test split - 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, 
    random_state= 32 # this is for reproducibility purposes
)
# no validation set because we will include this in the grid search

# scale the split data
scaler = StandardScaler()

# ******FIRST DATASET******
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# PCA dataset creation
# scale first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # scale on the unscaled dataset
X_test_scaled  = scaler.transform(X_test)

# PCA transformation to the scaled dataset
pca = PCA()  
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

# make them into dataframes
pca_cols = [f"PC{i+1}" for i in range(X_train_pca.shape[1])]

X_train_pca = pd.DataFrame(X_train_pca, columns=pca_cols, index=X_train.index)
X_test_pca  = pd.DataFrame(X_test_pca, columns=pca_cols, index=X_test.index)

# *****SECOND DATASET*****
# running on the first 7 PC's- since they explain about 75% of the variance
X_train_pca_selected = X_train_pca.iloc[:, :7]
X_test_pca_selected  = X_test_pca.iloc[:, :7]


# Third dataset creation
# remove 0 variance features and highly correlated features
from sklearn.feature_selection import VarianceThreshold

# set the threshold to 0.01, if its less than that then we remove
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X)

# get list of retained column names
cols_vt = X.columns[vt.get_support()]
X_vt_df = pd.DataFrame(X_vt, columns=cols_vt)

# remove the highly correlated features
corr = X_vt_df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
print(to_drop)

X_reduced = X_vt_df.drop(columns=to_drop)

# split scale run PCA on this reduced features
X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_red_scaled = scaler.fit_transform(X_train_red)
X_test_Red_scaled  = scaler.transform(X_test_red)

pca = PCA()
X_train_pca_red = pca.fit_transform(X_train_red_scaled)
X_test_pca_red  = pca.transform(X_test_Red_scaled)

# *****THIRD DATASET*****
# run on the first 7 PC's like before
X_train_red_selected = X_train_pca_red[:, :7]
X_test_red_selected  = X_test_pca_red[:, :7]

# Convert PCA arrays to DataFrames so SHAP can use .sample()
pc_names = [f"PC{i+1}" for i in range(7)]

X_train_red_selected = pd.DataFrame(X_train_red_selected, columns=pc_names)
X_test_red_selected  = pd.DataFrame(X_test_red_selected, columns=pc_names)


In [ ]:
# see what features contribute to PC2
# Get PCA loadings (components)
loadings = pd.DataFrame(
    pca.components_,
    columns=X_train.columns,
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# Sort PC2 features by absolute contribution, but keep actual signed loadings
pc2_sorted = loadings.loc["PC2"].reindex(
    loadings.loc["PC2"].abs().sort_values(ascending=False).index
)

print(pc2_sorted.head(10))  # top 10 features
# PCA loadings
loadings = pd.DataFrame(
    pca.components_,
    columns=X_train.columns,
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# Sort by absolute loading values (strongest contributors)
pc2_sorted = loadings.loc["PC2"].sort_values(key=lambda x: np.abs(x), ascending=False)

# Take top 10 contributors
top_pc2 = pc2_sorted.head(10)

# Plot
plt.figure(figsize=(8, 6))
top_pc2[::-1].plot(kind="barh")   # reverse for descending order visually
plt.title("Top Feature Contributions to PC2")
plt.xlabel("Loading Weight")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# See what features contribute to PC2 for the Reduced dataset
# get loadings
loadings_red = pd.DataFrame(
    pca.components_,
    columns=X_train_red.columns,   # original reduced feature names
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)

# print
pc2_loadings = loadings_red.loc["PC2"].sort_values(key=lambda x: abs(x), ascending=False)
print("PC2 Loadings (Reduced-Feature PCA):")
print(pc2_loadings.head(15))   # top 15 contributors

# plot
plt.figure(figsize=(8, 6))
pc2_loadings.head(15).plot(kind='barh')
plt.xlabel("Loading Weight")
plt.title("Top PC2 Loadings (Reduced-Feature PCA)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Random Forest Regression

In [ ]:
    # Defining the fixed parameters of RFR model
rfr_model = RandomForestRegressor(random_state= 8, oob_score = True)

    # Parameter grid to be tested in GridSearchCV
rfr_param_grid = { 'criterion':['squared_error', 'absolute_error', 'friedman_mse'],
                   'max_features' : [None, 'sqrt', 'log2'] }

parameter_tests = GridSearchCV(
    estimator=rfr_model,
    param_grid=rfr_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2)

In [ ]:
# *** FIRST DATASET VARIANT ***
parameter_tests.fit(X_train_scaled, y_train)

    # best model scoring metrics
print("Best R²:", parameter_tests.best_score_)
print("Best Params:", parameter_tests.best_params_)

    # applying best model to test set
optimized_rfr_model = parameter_tests.best_estimator_
rfr_y_pred = optimized_rfr_model.predict(X_test_scaled)

    # model performance
print("Test R²:", r2_score(y_test, rfr_y_pred))
print("Test MAE:", mean_absolute_error(y_test, rfr_y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, rfr_y_pred)))

    # creating SHAP summary
rfr_explainer = shap.TreeExplainer(optimized_rfr_model)
rfr_shap_values = rfr_explainer.shap_values(X_test_scaled)
shap.summary_plot(rfr_shap_values, X_test_scaled, feature_names=X.columns, show = False)
plt.tight_layout()
#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_shap_sum.png",bbox_inches='tight', dpi=300)

In [ ]:
# *** SECOND DATASET VARIANT***
parameter_tests.fit(X_train_pca_selected, y_train)

    # best model scoring metrics
print("Best R²:", parameter_tests.best_score_)
print("Best Params:", parameter_tests.best_params_)

    # applying best model to test set
pca_optimized_rfr_model = parameter_tests.best_estimator_
pca_rfr_y_pred = pca_optimized_rfr_model.predict(X_test_pca_selected)

    # model performance
print("Test R²:", r2_score(y_test, pca_rfr_y_pred))
print("Test MAE:", mean_absolute_error(y_test, pca_rfr_y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, pca_rfr_y_pred)))

    # generating SHAP summary
rfr_explainer = shap.TreeExplainer(pca_optimized_rfr_model)
rfr_shap_values = rfr_explainer.shap_values(X_test_pca_selected)
shap.summary_plot(rfr_shap_values, X_test_pca_selected, feature_names=X_train_pca_selected.columns, show = False)
plt.tight_layout()
#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_LPS_shap_sum.png",bbox_inches='tight', dpi=300)

In [ ]:
# *** THIRD DATASET VARIANT***
parameter_tests.fit(X_train_red_selected, y_train)

   # best model scoring metrics
print("Best R²:", parameter_tests.best_score_)
print("Best Params:", parameter_tests.best_params_)

    # applying best model to test set
red_pca_optimized_rfr_model = parameter_tests.best_estimator_
pca_red_rfr_y_pred = red_pca_optimized_rfr_model.predict(X_test_red_selected)

    # model performance
print("Test R²:", r2_score(y_test, pca_red_rfr_y_pred))
print("Test MAE:", mean_absolute_error(y_test, pca_red_rfr_y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, pca_red_rfr_y_pred)))

    # generating SHAP summary
rfr_explainer = shap.TreeExplainer(red_pca_optimized_rfr_model)
rfr_shap_values = rfr_explainer.shap_values(X_test_red_selected)
shap.summary_plot(rfr_shap_values, X_test_red_selected, feature_names=X_train_red_selected.columns, show = False)
plt.tight_layout()
#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_LPS_red_shap_sum.png",bbox_inches='tight', dpi=300)

In [ ]:
# generating residuals plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.scatterplot(x=rfr_y_pred, y=y_test, color = "olive", ax = axes[0])
axes[0].set_xlabel("Predicted Mean Yield")
axes[0].set_ylabel("Residuals")
axes[0].set_title('Log Trans. and Scaled RFR Residuals')
axes[0].grid(True)

sns.residplot(x=pca_rfr_y_pred, y=y_test, color = 'magenta', ax = axes[1])
axes[1].set_xlabel("Predicted Mean Yield")
axes[1].set_ylabel("Residuals")
axes[1].set_title(' Scaled, Log + PCA Trans RFR Residuals')
axes[1].grid(True)

sns.residplot(x=pca_red_rfr_y_pred, y=y_test,color = 'blue', ax = axes[2])
axes[2].set_xlabel("Predicted Mean Yield")
axes[2].set_ylabel("Residuals")
axes[2].set_title(' Scaled, Log + PCA Reduced RFR Residuals')
axes[2].grid(True)

#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_resid_multi.png")
plt.show()

In [ ]:
# generating QQ-plots
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sm.qqplot(data=(y_test - rfr_y_pred), line='45',ax = axes[0])
axes[0].set_title('Log Trans. and Scaled Residuals QQ Plot')

sm.qqplot(data=(y_test - pca_rfr_y_pred), line='45', ax = axes[1])
axes[1].set_title('Scaled, Log + PCA Trans Residuals QQ Plot')

sm.qqplot(data=(y_test - pca_red_rfr_y_pred), line='45', ax = axes[2])
axes[2].set_title('Scaled, Log + PCA Reduced Residuals QQ Plot')

#plt.savefig(r"/Users/adia/PyCharmMiscProject/ML/Final/Figures/RFR_qq_multi.png")
plt.show()

### Support Vector Regression

In [ ]:
# Support Vector Regression Code
# scale the data and ft the model
# ***** First Dataset*****
svr_model = Pipeline([ # use pipleline to scale and train the data
    ("scaler", StandardScaler()), # scale after splitting to prevent data leakage!
    ("svr", SVR(kernel= "rbf")) # chose RBF kernel since the relationship between X and y is most likely non linear
])
#hyperparameter grid search for the best parameters
param_grid = {
    "svr__C": [0.1, 1, 10, 100],
    "svr__gamma": ["scale", "auto", 0.01, 0.001, 0.0001],
    "svr__epsilon": [0.1, 0.5, 1.0]
}

grid = GridSearchCV(
    estimator= svr_model,
    param_grid= param_grid,
    cv=5, # cross validation
    scoring="r2",
    n_jobs=-1, 
    verbose= 2
)

# fit this grid only on the training data
grid.fit(X_train, y_train) # since we are scaling in the pipeline we won't use the scaled data

print("Best R² training:", grid.best_score_)
print("Best Params:", grid.best_params_)

best_svr_model = grid.best_estimator_ # assign out best

# evaluate on the test set
y_pred = grid.predict(X_test)

print("Test R²:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

# SHAP figure code
explainer = shap.KernelExplainer(grid.predict, X_train.sample(50))
shap_values = explainer.shap_values(X_test.sample(50))

shap.summary_plot(shap_values, X_test.sample(50))

#residuals Code
import statsmodels as sm
residuals = y_test - y_pred

# residuals
sns.residplot(x=y_pred, y=residuals, lowess=True)
plt.xlabel("Predicted Mean Yield")
plt.ylabel("Residuals")
plt.title("SVR Residual Plot (Log-Transformed & Scaled Data)")
plt.grid(True)
plt.tight_layout()
plt.savefig("svr_residual_plot.png", dpi=300)
plt.show()

# qq
sm.qqplot(residuals, line='45')
plt.title("SVR Residuals QQ Plot (Log-Transformed & Scaled Data)")
plt.tight_layout()
plt.savefig("svr_qq_plot.png", dpi=300)
plt.show()

# *****Second Dataset*****
# fit the model
svr_model_pca = SVR(kernel="rbf")

#hyper parameters
param_grid_pca = {
    "C": [1, 10, 100, 1000, 10000],
    "gamma": [1, 0.5, 0.1, 0.01, 0.001],
    "epsilon": [0.001, 0.01, 0.1]
}

# grid search
grid = GridSearchCV(
    estimator=svr_model_pca, 
    param_grid= param_grid_pca, 
    cv= 5, 
    scoring = "r2", 
    n_jobs=1, 
)
# fit this grid only on the training data
grid.fit(X_train_pca_selected, y_train)
# PCA is only for the predictors

print("Best R²:", grid.best_score_)
print("Best Params:", grid.best_params_)

best_svr_pca = grid.best_estimator_

# evaluate on the test set
y_pred_pca = grid.predict(X_test_pca_selected)

print("Test R²:", r2_score(y_test, y_pred_pca))
print("Test MAE:", mean_absolute_error(y_test, y_pred_pca))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_pca)))

# shap values on the PCA set: 
explainer = shap.KernelExplainer(grid.predict, X_train_pca_selected.sample(50))
shap_values = explainer.shap_values(X_test_pca_selected.sample(50))

shap.summary_plot(shap_values, X_test_pca_selected.sample(50))
# residuals for PCA model
residuals_pca = y_test - y_pred_pca

# residual
sns.residplot(x=y_pred_pca, y=residuals_pca, lowess=True)
plt.xlabel("Predicted Mean Yield (PCA SVR)")
plt.ylabel("Residuals")
plt.title("SVR Residual Plot (PCA Transformed Data)")
plt.grid(True)
plt.tight_layout()
plt.savefig("svr_pca_residual_plot.png", dpi=300)
plt.show()

# qq
sm.qqplot(residuals_pca, line='45')
plt.title("SVR Residuals QQ Plot (PCA Transformed Data)")
plt.tight_layout()
plt.savefig("svr_pca_residual_qq_plot.png", dpi=300)
plt.show()

# *****Third Dataset*****
# fit the model
svr_model_pca_red = SVR(kernel="rbf")

#hyper parameters
param_grid_pca_red = {
    "C": [10, 100, 1000, 10000], 
    "gamma": [ 1, 0.1, 0.01, 0.001], 
    "epsilon": [0.01, 0.05, 0.1]
}

# grid search
grid = GridSearchCV(
    estimator=svr_model_pca_red, 
    param_grid= param_grid_pca_red, 
    cv= 5, 
    scoring = "r2", 
    n_jobs=1, 
)
# fit this grid only on the training data
grid.fit(X_train_red_selected, y_train)
# PCA is only for the predictors

print("Best R²:", grid.best_score_)
print("Best Params:", grid.best_params_)

best_svr_pca = grid.best_estimator_
# evaluate on the test set
y_pred_red = grid.predict(X_test_red_selected)

print("Test R²:", r2_score(y_test, y_pred_red))
print("Test MAE:", mean_absolute_error(y_test, y_pred_red))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_red)))

#Residuals
# Calculate residuals
residuals_red = y_test - y_pred_red

# residual
sns.residplot(x=y_pred_red, y=residuals_red, lowess=True)
plt.xlabel("Predicted Mean Yield (PCA Reduced SVR)")
plt.ylabel("Residuals")
plt.title("SVR Residual Plot (Reduced and PCA Transformed Data)")
plt.grid(True)
plt.tight_layout()
plt.savefig("svr_pca_residual_plot.png", dpi=300)
plt.show()


# shap on the features
explainer = shap.KernelExplainer(grid.predict, X_train_red_selected.sample(50))
shap_values = explainer.shap_values(X_test_red_selected.sample(50))

shap.summary_plot(shap_values, X_test_red_selected.sample(50))
